<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [ ]:
%pip install llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.8/443.8 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


In [ ]:
from llama_cpp import Llama
from tqdm import tqdm
from transformers import AutoTokenizer
import requests
from collections import defaultdict
import json

In [ ]:
# Load the model
llm = Llama.from_pretrained(repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", # repository name
                            filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf", # model file
                            n_gpu_layers=-1, # use all GPU layers
                            n_ctx=32768, # context size
                            flash_attn=True, # use flash attention
                            chat_format="llama-3", # chat format
                            verbose=False)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


./Meta-Llama-3.1-8B-Instruct-Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

In [ ]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/ticket_to_ride.txt').text
rulebook[:100]

In [ ]:
# Check that input is inside context window
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
tokens = tokenizer.encode(rulebook)
print(len(tokens))

In [ ]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [ ]:
def multiple_model_test(models, prompts, games, iterations):
    outputs = defaultdict(dict)

    for model,game,prompt,prompt_idx,it in tqdm([(m,f,p,idx,it) for f in game_names for (idx,p) in enumerate(prompts) for m in models for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        outputs[model][prompt_idx][game+'-'+it] = llm.create_chat_completion(
            messages= generate_message(prompt, rulebook),
            temperature=0.7,
        )['choices'][0]['message']['content']

    with open('extraction.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

100%|██████████| 8/8 [02:15<00:00, 16.94s/it]


# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [ ]:
prompts = ["""You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation""",
           """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g., “like a treasure hunt” or “like building a LEGO city”).
           Use the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n""",
]
game_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']
models = [llm]
iterations = 1

multiple_model_test(models, prompts, game_names, iterations)

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 1 -> **unsolvable** mechanic that uses piece not presents in the game (every time you play a card you can steal a warrior token from every opponent but there is no way to obtain a warrior token)
- level 2 -> **unsolvable** (in one line states you can draw two cards, in another one that you can draw only one)
- level 3 -> **incoherent** and hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4 -> **coherent** but obviously gamebreaking (draw infinite cards each turn)
- level 5 -> **coherent** but very unbalanced (the first player can play two turns)

In [ ]:
prompts = ["""test"""]
game_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']
models = [llm]
iterations = 5

multiple_model_test(models, prompts, game_names, iterations)